# CME Futures: Stochastic Discount-Factor Features

The stochastic discount-factor model learns fold-scoped latent factors from the product panel and
maps them to each declared forward-return horizon. Training rows determine the representation;
validation rows are transformed without refitting. The fitted model, fold identity, prediction
shard, and eligible validation keys are persisted together.

The notebook executes the declared SDF configurations and publishes their catalog rows. IC remains
diagnostic. The equal-weight validation backtest in `13_backtest` selects configurations.

One framing to carry through the rest: this is the most theoretically motivated model in the
case study and it is given no special standing because of that. Its rows enter the same funnel
as the linear model's, are selected on the same statistic, and can lose to a gradient-boosting
configuration that rests on no asset-pricing argument at all. A better story about why a model
should work is not evidence that it does.

Prerequisites: `03_financial_features`, `04_model_based_features`, and `05_evaluation`.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# CME 期貨：隨機折現因子特徵

隨機折現因子模型從商品面板學習各資料折專屬的潛在因子，並將其對應至每種已宣告的未來報酬預測期。訓練資料列決定表示方式；驗證資料列只經轉換，不重新擬合。擬合後模型、資料折識別資訊、預測分片及合格的驗證鍵值會一併保存。

本 Notebook 執行已宣告的隨機折現因子設定，並發布其目錄資料列。資訊係數仍只作診斷。`13_backtest` 的等權重驗證回測負責選擇設定。

閱讀後續內容時應記住一項原則：這是本案例研究中理論動機最強的模型，但不因此享有特殊地位。它的資料列與線性模型進入同一選擇流程，依相同統計量選擇，也可能輸給完全不依賴資產定價論點的梯度提升設定。模型應該有效的理由說得更好，不等於有證據證明它有效。

先決條件：`03_financial_features`、`04_model_based_features` 與 `05_evaluation`。
<!-- ML4T_ZH_END c0000-p000 -->


## What a stochastic discount factor is

Asset pricing has one central result worth stating plainly, because everything this notebook
does follows from it. If prices admit no arbitrage, then there exists a single random variable
- call it `m` - such that for every asset, the expected product of `m` and that asset's return
is the same constant. One object prices everything: equities, bonds, corn futures, all of it.
That object is the **stochastic discount factor**.

Its usefulness is that it converts "which assets earn more, and why" into a question about one
quantity. An asset earns a premium when its return covaries negatively with `m` - when it pays
badly in the states `m` says are expensive, which are the states investors care most about
being protected in. Under that view a risk premium is not compensation for variance; it is
compensation for failing to pay off when payment matters.

`m` is not observable. But it is a well-defined thing to estimate, and estimating it with a
neural network means not having to assume in advance what functional form it takes - which
matters, because the classical models that assume one (a market factor, a small set of
characteristics entering linearly) have a long record of being rejected on the data.

## How this differs from `10a`, on two axes rather than one

The index for this stage frames the two configurations as differing in objective. That is true
and it is not the whole difference - they also see different data, which is worth being exact
about because it changes what each is capable of finding.

**PCA sees returns only.** `run_pca_fold` takes the characteristics panel and discards it with
`del` before fitting; it is handed a matrix of product returns and nothing else.


<!-- ML4T_ZH_BEGIN c0001-p000 -->
## 什麼是隨機折現因子

資產定價有一項核心結果值得直接說明，因為本 Notebook 的所有工作都由此出發。若價格不存在套利，就存在一個單一隨機變數，記為 `m`，使得對每項資產，`m` 與該資產報酬乘積的期望值，都等於同一常數。同一個對象為所有資產定價：股票、債券、玉米期貨，無一例外。這個對象就是**隨機折現因子**。

它的用途，是將「哪些資產賺得較多，以及為什麼」轉化為關於單一數量的問題。某資產若報酬與 `m` 呈負共變異，就會賺取溢酬；也就是說，它在 `m` 指出價格昂貴的狀態下表現不佳，而那正是投資人最希望得到保障的狀態。從這個角度看，風險溢酬不是對變異數的補償，而是對最需要給付時卻無法給付的補償。

`m` 無法直接觀察，但它是定義明確的估計對象。用神經網路估計，便無須事先假定其函數形式。這點很重要，因為採用特定函數形式的傳統模型，例如市場因子或線性納入的一小組特徵，長期以來屢遭資料否定。

## 它與 `10a` 在兩個面向上有何差異

本階段的索引將兩項設定描述為目標不同。這固然正確，卻不是全部差異；它們看到的資料也不同。這點值得精確說明，因為它會改變各自能找到的資訊。

**PCA 只看到報酬。**`run_pca_fold` 取得特徵面板，在擬合前以 `del` 將其捨棄；它只取得商品報酬矩陣，沒有其他資料。
<!-- ML4T_ZH_END c0001-p000 -->

**The SDF sees returns and characteristics.** `run_sdf_fold` passes `chars_train` through to
`_cross_section_batch(chars_train, returns=returns_train)`, and the number of instruments it
builds is derived from the characteristics' width. So carry, momentum and volatility are
inputs here in a way they are not in `10a`.

The two differences compound. PCA asks which directions explain the most variation in returns,
using returns alone. The SDF asks which combination of assets, weighted by their observable
characteristics, best explains the cross-section of returns - so it can express "products with
high carry and low volatility load on this factor" in a way PCA structurally cannot, because
PCA never sees carry.

That is why the comparison between them is informative rather than decorative. A win for the
SDF is evidence that the characteristics carry pricing information beyond what the return
covariance already encodes. A win for PCA is evidence that they do not, and that the extra
freedom bought overfitting.

<!-- ML4T_ZH_BEGIN c0001-p001 -->
**隨機折現因子看到報酬與特徵。**`run_sdf_fold` 將 `chars_train` 傳給 `_cross_section_batch(chars_train, returns=returns_train)`，所建立的工具變數數量取決於特徵的寬度。因此，持有收益、動能及波動率會成為此處的輸入，與 `10a` 不同。

這兩項差異會相互加強。PCA 只使用報酬，問的是哪些方向能解釋最多報酬變動；隨機折現因子則問依可觀察特徵加權的哪種資產組合，最能解釋報酬的橫斷面。因此它能表達「持有收益高、波動率低的商品對此因子有負荷」，PCA 在結構上做不到，因為它從未看到持有收益。

因此，兩者的比較有實質資訊價值。若隨機折現因子勝出，便有證據顯示特徵包含超出報酬共變異數所編入的定價資訊。若 PCA 勝出，則有證據顯示特徵不包含這類額外資訊，而增加的自由度造成過度擬合。
<!-- ML4T_ZH_END c0001-p001 -->


In [1]:
"""Fit the declared CME futures stochastic discount-factor population."""

import polars as pl

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    model_request_catalog,
    open_study,
    product_universe_table,
    resolve_model_requests,
    resolved_model_plan,
    run_official_model_catalog,
    run_resolved_model_requests,
)

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None
PREVIEW_REDUCTIONS: dict = {}
# The population hash this run replaces, read from the registry and set by a person. A
# first population takes None; a re-run whose membership has changed is refused without
# the hash it supersedes, and the refusal names the value required.
SUPERSEDES_POPULATION: str | None = None

## Declared requests

Both return horizons use the named stochastic discount-factor configuration. The resolved plan
shows the eligible rows, folds, feature count, checkpoint schedule, and identity before fitting.

**This model publishes on `cuda`, declared in `setup.yaml` rather than detected.** The device
enters both `runtime` and `numerical_runtime` inside the hashed computation, so leaving it to
`preferred_latent_device()` would resolve one training identity on a GPU host and a different one
on a CPU host, both publishing under this population's name. `10a_pca` overrides the same
declaration to `cpu`, because PCA has no GPU implementation to record.

**Nothing here fails on a fit that did not converge.** The shared runner enforces convergence for
IPCA only, and this case study does not declare IPCA; for the stochastic discount factor it
records the training history and the terminal Sharpe as fold extras and checks neither. That is a
gap in shared code rather than in this notebook, so what is claimed below is that every declared
checkpoint was produced and registered - not that the objective had settled when it was.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 已宣告的訓練請求

兩種報酬預測期都使用具名的隨機折現因子設定。解析後的計畫會在擬合前，顯示合格資料列、資料折、特徵數量、檢查點排程與識別資訊。

**此模型在 `cuda` 上發布，該裝置是在 `setup.yaml` 中宣告，而非自動偵測。**裝置會進入雜湊運算內的 `runtime` 與 `numerical_runtime`，因此若交由 `preferred_latent_device()` 決定，就會在 GPU 主機與 CPU 主機上解析出不同訓練識別資訊，卻都以此模型群組的名稱發布。`10a_pca` 將同一宣告覆寫為 `cpu`，因為 PCA 沒有可記錄的 GPU 實作。

**擬合未收斂時，此處不會因此判定失敗。**共用執行器只對 IPCA 強制檢查收斂，而本案例研究並未宣告 IPCA。對隨機折現因子，執行器將訓練歷程與最後的夏普比率記錄為資料折附加資訊，但不檢查兩者。這是共用程式碼的缺口，而非本 Notebook 的缺口；因此下方所主張的，只是每個已宣告檢查點都已產生並登錄，並不表示目標函式當時已經收斂。
<!-- ML4T_ZH_END c0004-p000 -->


In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE)
requests = model_request_catalog(
    "latent_factors",
    labels=ALL_LABELS,
    config_names=("sdf",),
)
resolved = resolve_model_requests(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    preview_reductions=PREVIEW_REDUCTIONS,
)
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


In [4]:
resolved_model_plan(resolved)

family,label,config_name,task,feature_count,eligible_entities,eligible_rows,folds,validation_start,validation_end,checkpoints,execution_tier,training_hash
str,str,str,str,i64,i64,i64,i64,datetime[μs],datetime[μs],i64,str,str
"""latent_factors""","""fwd_ret_21d""","""sdf""","""regression""",69,30,37782,5,2019-01-03 00:00:00,2023-11-29 00:00:00,5,"""canonical""","""7e0f27986e67"""
"""latent_factors""","""fwd_ret_5d""","""sdf""","""regression""",69,30,38262,5,2019-01-03 00:00:00,2023-12-21 00:00:00,5,"""canonical""","""b08cb6530703"""


## Execute and validate

The shared latent-factor runner fits each representation inside its training fold, persists the
fitted state, and requires the complete validation key set before publication.

### Why checkpoints are part of what gets selected

A neural fit is not one model, it is a trajectory. It passes through a sequence of states as
training proceeds, and which one is kept is a decision with the same standing as the choice of
architecture. `checkpoint_epochs` declares the epochs this configuration publishes, so each
becomes its own candidate row, and `13_backtest` selects among them on validation backtest
Sharpe like any other configuration.

Publishing them rather than picking one is the honest arrangement. Choosing the best epoch by
looking at validation performance and then reporting that model's validation performance is
selection inside the number being reported, and it does not stop being that because the choice
was made by hand. Declaring the checkpoints up front puts the choice into the same funnel and
the same trial count as everything else - which the deflated Sharpe downstream then divides by.

This is the concrete contrast with `10a`, which has one fitted state per fold and nothing to
checkpoint: its solution is closed-form, so there is no trajectory to choose a point on.

### What the fold discipline protects

Training rows determine the representation and validation rows are transformed without
refitting. For a model that reads characteristics as well as returns, that discipline covers
two channels rather than one: the factors must not be shaped by validation returns, and the
instrument weights must not be shaped by validation characteristics either. Both would produce
predictions that look ordinary and a backtest that runs.


<!-- ML4T_ZH_BEGIN c0007-p000 -->
## 執行與驗證

共用潛在因子執行器會在各自的訓練折內擬合每種表示方式、保存擬合狀態，並要求完整的驗證鍵值集合，才能發布結果。

### 為何檢查點是選擇對象的一部分

神經網路擬合並非單一模型，而是一條軌跡。訓練過程會經過一連串狀態，保留哪個狀態，是與選擇架構同等重要的決策。`checkpoint_epochs` 宣告此設定要發布的訓練週期，因此每個檢查點都成為獨立候選資料列，`13_backtest` 再像選擇其他設定一樣，依驗證回測夏普比率從中選擇。

發布檢查點，而非事先挑出一個，才是如實呈現的安排。先查看驗證表現、選出最佳訓練週期，再報告該模型的驗證表現，等於在所報數值內部進行選擇；即使人工選擇也是如此。預先宣告檢查點，能使這項選擇與其他設定進入同一選擇流程和試驗次數計算；後續的縮減夏普比率再依該次數進行調整。

這與 `10a` 形成具體對比。後者每折只有一個擬合狀態，沒有檢查點；它使用封閉形式解，不存在可從中挑選某個時點的軌跡。

### 資料折紀律保護了什麼

訓練資料列決定表示方式，驗證資料列只經轉換，不重新擬合。對同時讀取特徵與報酬的模型而言，這項紀律涵蓋兩個管道，而非一個：因子不得受驗證報酬影響，工具變數權重也不得受驗證特徵影響。兩種洩漏都會產生看似正常的預測及能順利執行的回測。
<!-- ML4T_ZH_END c0007-p000 -->

### Why the complete key set is required

The same reason it is required in `10a`, and for the same structural cause: this is a
cross-sectional model, so a missing product does not leave one prediction absent - it changes
the estimated factors and therefore every prediction the fold publishes. An incomplete key set
is refused rather than published short.

### Where the freedom cuts against it

The SDF has far more capacity than the eigenvectors of a covariance matrix, and this panel is
thirty products. Capacity on a panel that size is as likely to fit noise as structure, and
nothing in a training loss distinguishes the two. That is what makes `10a` the baseline worth
beating rather than a formality - and what makes a narrow SDF win, if one appears, weaker
evidence than its margin suggests.

<!-- ML4T_ZH_BEGIN c0007-p001 -->
### 為何要求完整鍵值集合

原因與 `10a` 相同，結構性原因也相同：這是橫斷面模型，因此缺少一項商品，不只是缺一項預測，還會改變估計的因子，進而改變該折發布的每項預測。鍵值集合不完整時會遭拒絕，不會發布缺漏的結果。

### 自由度在哪裡反過來不利於模型

隨機折現因子的容量，遠高於共變異數矩陣的特徵向量，而此面板只有 30 項商品。在這種規模的面板上，容量擬合雜訊的可能性不亞於擬合結構，訓練損失本身無法區分兩者。這使 `10a` 成為真正值得超越的基準，而非形式上的對照；若隨機折現因子僅小幅勝出，證據的力度也會弱於表面差距所暗示的程度。
<!-- ML4T_ZH_END c0007-p001 -->


In [5]:
if EXECUTION_TIER == "canonical":
    execution, population = run_official_model_catalog(
        study,
        requests,
        population_name="cme_futures-sdf-validation-v1",
        resolved_requests=resolved,
        supersedes=SUPERSEDES_POPULATION,
    )
else:
    if WORKSPACE is None or not PREVIEW_REDUCTIONS:
        raise ValueError("preview execution requires WORKSPACE and PREVIEW_REDUCTIONS")
    execution = run_resolved_model_requests(study, resolved)
    population = None

Latent factor CV: 1 models × 5 folds
Log file: case_studies/cme_futures/run_log/training/b08cb6530703/.models.844d0f663452444b8e8ec289412be1a2.tmp/latent_factors.log
Scoring: dates=all cadence=- step=1 checkpoint_selection=fixed reporting_epoch=1280
  sdf (K=5):


    Fold 0: ragged train=2039, val=258, max_N=30


      fold 0: reported_epoch=1280, IC=+0.0320, 476.7s


    Fold 1: ragged train=2059, val=258, max_N=30


      fold 1: reported_epoch=1280, IC=+0.0028, 555.2s


    Fold 2: ragged train=2059, val=258, max_N=30


... (14 outputs left out)

      fold 3: reported_epoch=1280, IC=+0.0801, 920.9s


    Fold 4: ragged train=2043, val=237, max_N=30


      fold 4: reported_epoch=1280, IC=-0.0577, 887.6s


    -> reporting epoch=1280 (fixed), IC=+0.0369 (4930.2s)
  Best: sdf (IC=+0.0369)


In [6]:
catalog = execution.catalog_rows.select(
    "family",
    "label",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "execution_tier",
    "complete",
    "training_hash",
    "prediction_hash",
).sort("label", "checkpoint_value")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("stochastic discount-factor execution returned a partial prediction")
catalog

family,label,config_name,checkpoint_kind,checkpoint_value,execution_tier,complete,training_hash,prediction_hash
str,str,str,str,i64,str,bool,str,str
"""latent_factors""","""fwd_ret_21d""","""sdf""","""epoch""",256,"""canonical""",true,"""7e0f27986e67""","""a3d67442f32e"""
"""latent_factors""","""fwd_ret_21d""","""sdf""","""epoch""",512,"""canonical""",true,"""7e0f27986e67""","""4aa2fc4222d1"""
"""latent_factors""","""fwd_ret_21d""","""sdf""","""epoch""",768,"""canonical""",true,"""7e0f27986e67""","""430e5789167d"""
"""latent_factors""","""fwd_ret_21d""","""sdf""","""epoch""",1024,"""canonical""",true,"""7e0f27986e67""","""d2c0652f07cf"""
"""latent_factors""","""fwd_ret_21d""","""sdf""","""epoch""",1280,"""canonical""",true,"""7e0f27986e67""","""a3eec072c49f"""
"""latent_factors""","""fwd_ret_5d""","""sdf""","""epoch""",256,"""canonical""",true,"""b08cb6530703""","""5329aac81fdc"""
"""latent_factors""","""fwd_ret_5d""","""sdf""","""epoch""",512,"""canonical""",true,"""b08cb6530703""","""8ca7dd2abeff"""
"""latent_factors""","""fwd_ret_5d""","""sdf""","""epoch""",768,"""canonical""",true,"""b08cb6530703""","""21571c7d1e9d"""
"""latent_factors""","""fwd_ret_5d""","""sdf""","""epoch""",1024,"""canonical""",true,"""b08cb6530703""","""fdf6420edfcb"""
